In [0]:
print("Welcome to Databricks!")
print("My Data Engineering portfolio starts today.")

spark.sql("SELECT current_catalog() AS catalog").show()

spark.sql("SHOW CATALOGS").show()

spark.sql("CREATE CATALOG IF NOT EXISTS sinthuja_portfolio")

catalog = "sinthuja_portfolio"
schema = "ecommerce_portfolio"

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`"
)

spark.sql(
    f"CREATE VOLUME IF NOT EXISTS "
    f"`{catalog}`.`{schema}`.`landing`"
)

volume_path = f"/Volumes/{catalog}/{schema}/landing"
print(volume_path)


orders_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(f"{volume_path}/orders.csv")
)

display(orders_df.limit(10))


from pyspark.sql.functions import current_timestamp

datasets = [
    "customers", "products", "orders",
    "order_items", "payments", "returns"
]

for name in datasets:
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .csv(f"{volume_path}/{name}.csv")
        .withColumn("_ingested_at", current_timestamp())
    )

    table_name = f"`{catalog}`.`{schema}`.`bronze_{name}`"

    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(table_name))

    print(name, df.count())














customers 24
products 18
orders 49
order_items 96
payments 48
returns 3
